# Likelihood Curves for Parameter Estimation

This notebook creates likelihood curves for each parameter in the model, holding all other parameters constant.
For vector parameters (like γ_p and σ_p), users can specify an index to select a specific pulsar.

Updated to work with the current argus codebase structure.

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add the python directory to the path
sys.path.append('../python')

from argus import data_loader, workflow, bayesian_inference
from argus.jax_kalman_filter import JaxKalmanFilter
import configparser

## Setup: Load Data and Initialize Kalman Filter

We'll use the configuration system to set up our model consistently with the inference code.

In [ ]:
def setup_kalman_filter_from_config(config_path="../python/argus/configs/config_numpyro_test_010.ini"):
    """Set up Kalman filter using the same workflow as the main inference code."""
    
    # Load configuration
    config = configparser.ConfigParser()
    config.read(config_path)
    
    # Use the actual workflow function
    pulsar_data, KF = workflow.setup_data_and_kalman_filter(config, logger=None, use_gw=True)
    
    # Get noise parameters
    efac_array, equad_array, sigma_p_array, gamma_p_array = workflow.get_noise_parameters(config)
    
    return KF, pulsar_data, efac_array, equad_array, sigma_p_array, gamma_p_array, config

# Setup the model
print("Setting up Kalman filter and loading data...")
KF, pulsar_data, efac_array, equad_array, sigma_p_array, gamma_p_array, config = setup_kalman_filter_from_config()
n_pulsars = len(pulsar_data['metadata'])
print(f"Loaded data for {n_pulsars} pulsars")

## Likelihood Curve Functions

These functions create likelihood curves for different parameter types.

In [ ]:
def create_baseline_parameters():
    """Create baseline parameter values for likelihood curves."""
    return bayesian_inference.Parameters(
        γa=1e-9,  # Fixed GW spectral index
        ha=1e-15,  # GW amplitude baseline
        γp=gamma_p_array,  # Pulsar red noise gamma
        σp=sigma_p_array,  # Pulsar red noise sigma  
        EFAC=efac_array,   # Error factors
        EQUAD=equad_array  # Extra quadrature noise
    )

def likelihood_curve_ha(ha_values):
    """Create likelihood curve for GW amplitude ha."""
    baseline_params = create_baseline_parameters()
    log_likelihoods = []
    
    print(f"Computing likelihood curve for ha with {len(ha_values)} points...")
    
    for i, ha in enumerate(ha_values):
        if i % 10 == 0:
            print(f"  Progress: {i+1}/{len(ha_values)}")
            
        params = baseline_params._replace(ha=ha)
        ll = KF.get_likelihood(params)
        log_likelihoods.append(float(ll))
    
    return jnp.array(log_likelihoods)

def likelihood_curve_gamma_p(gamma_p_values, pulsar_index=0):
    """Create likelihood curve for pulsar red noise gamma_p for a specific pulsar."""
    baseline_params = create_baseline_parameters()
    log_likelihoods = []
    
    print(f"Computing likelihood curve for γ_p[{pulsar_index}] with {len(gamma_p_values)} points...")
    
    for i, gamma_p in enumerate(gamma_p_values):
        if i % 10 == 0:
            print(f"  Progress: {i+1}/{len(gamma_p_values)}")
            
        # Modify only the specified pulsar's gamma_p
        gamma_p_modified = baseline_params.γp.at[pulsar_index].set(gamma_p)
        params = baseline_params._replace(γp=gamma_p_modified)
        ll = KF.get_likelihood(params)
        log_likelihoods.append(float(ll))
    
    return jnp.array(log_likelihoods)

def likelihood_curve_sigma_p(sigma_p_values, pulsar_index=0):
    """Create likelihood curve for pulsar red noise sigma_p for a specific pulsar."""
    baseline_params = create_baseline_parameters()
    log_likelihoods = []
    
    print(f"Computing likelihood curve for σ_p[{pulsar_index}] with {len(sigma_p_values)} points...")
    
    for i, sigma_p in enumerate(sigma_p_values):
        if i % 10 == 0:
            print(f"  Progress: {i+1}/{len(sigma_p_values)}")
            
        # Modify only the specified pulsar's sigma_p
        sigma_p_modified = baseline_params.σp.at[pulsar_index].set(sigma_p)
        params = baseline_params._replace(σp=sigma_p_modified)
        ll = KF.get_likelihood(params)
        log_likelihoods.append(float(ll))
    
    return jnp.array(log_likelihoods)

def likelihood_curve_efac(efac_values, pulsar_index=0):
    """Create likelihood curve for EFAC for a specific pulsar."""
    baseline_params = create_baseline_parameters()
    log_likelihoods = []
    
    print(f"Computing likelihood curve for EFAC[{pulsar_index}] with {len(efac_values)} points...")
    
    for i, efac in enumerate(efac_values):
        if i % 10 == 0:
            print(f"  Progress: {i+1}/{len(efac_values)}")
            
        # Modify only the specified pulsar's EFAC
        efac_modified = baseline_params.EFAC.at[pulsar_index].set(efac)
        params = baseline_params._replace(EFAC=efac_modified)
        ll = KF.get_likelihood(params)
        log_likelihoods.append(float(ll))
    
    return jnp.array(log_likelihoods)

def likelihood_curve_equad(equad_values, pulsar_index=0):
    """Create likelihood curve for EQUAD for a specific pulsar."""
    baseline_params = create_baseline_parameters()
    log_likelihoods = []
    
    print(f"Computing likelihood curve for EQUAD[{pulsar_index}] with {len(equad_values)} points...")
    
    for i, equad in enumerate(equad_values):
        if i % 10 == 0:
            print(f"  Progress: {i+1}/{len(equad_values)}")
            
        # Modify only the specified pulsar's EQUAD
        equad_modified = baseline_params.EQUAD.at[pulsar_index].set(equad)
        params = baseline_params._replace(EQUAD=equad_modified)
        ll = KF.get_likelihood(params)
        log_likelihoods.append(float(ll))
    
    return jnp.array(log_likelihoods)

## Test Likelihood Evaluation

First, let's test that our likelihood evaluation works correctly.

In [ ]:
# Test likelihood evaluation
baseline_params = create_baseline_parameters()
print("Testing likelihood evaluation...")

# First call for JIT compilation
ll_test = KF.get_likelihood(baseline_params)
ll_test.block_until_ready()
print(f"Baseline log likelihood: {float(ll_test):.2f}")
print("Likelihood evaluation successful!")

## 1. GW Amplitude (ha) Likelihood Curve

In [ ]:
# Define ha parameter range
ha_min, ha_max = 1e-17, 1e-14
ha_values = jnp.logspace(jnp.log10(ha_min), jnp.log10(ha_max), 50)

# Compute likelihood curve
ll_ha = likelihood_curve_ha(ha_values)

# Plot
plt.figure(figsize=(10, 6))
plt.semilogx(ha_values, ll_ha, 'b-', linewidth=2)
plt.xlabel('GW Amplitude ha')
plt.ylabel('Log Likelihood')
plt.title('Likelihood Curve for GW Amplitude')
plt.grid(True, alpha=0.3)

# Mark maximum
max_idx = jnp.argmax(ll_ha)
plt.axvline(ha_values[max_idx], color='r', linestyle='--', alpha=0.7, 
           label=f'Max at ha = {ha_values[max_idx]:.2e}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Maximum likelihood at ha = {ha_values[max_idx]:.2e}")
print(f"Log likelihood value: {ll_ha[max_idx]:.2f}")

## 2. Pulsar Red Noise Parameters

### 2a. γ_p Likelihood Curve for Selected Pulsar

In [ ]:
# User-configurable pulsar index
PULSAR_INDEX = 5  # Change this to select different pulsar

print(f"Creating γ_p likelihood curve for pulsar index {PULSAR_INDEX}")
print(f"Baseline γ_p value for this pulsar: {gamma_p_array[PULSAR_INDEX]:.2e}")

# Define gamma_p parameter range
gamma_p_baseline = gamma_p_array[PULSAR_INDEX]
gamma_p_min = gamma_p_baseline * 0.01  # 1% of baseline
gamma_p_max = gamma_p_baseline * 100   # 100x baseline
gamma_p_values = jnp.logspace(jnp.log10(gamma_p_min), jnp.log10(gamma_p_max), 50)

# Compute likelihood curve
ll_gamma_p = likelihood_curve_gamma_p(gamma_p_values, PULSAR_INDEX)

# Plot
plt.figure(figsize=(10, 6))
plt.semilogx(gamma_p_values, ll_gamma_p, 'g-', linewidth=2)
plt.xlabel(f'γ_p[{PULSAR_INDEX}] (s⁻¹)')
plt.ylabel('Log Likelihood')
plt.title(f'Likelihood Curve for Pulsar {PULSAR_INDEX} Red Noise γ_p')
plt.grid(True, alpha=0.3)

# Mark maximum and baseline
max_idx = jnp.argmax(ll_gamma_p)
plt.axvline(gamma_p_values[max_idx], color='r', linestyle='--', alpha=0.7,
           label=f'Max at γ_p = {gamma_p_values[max_idx]:.2e}')
plt.axvline(gamma_p_baseline, color='orange', linestyle=':', alpha=0.7,
           label=f'Baseline γ_p = {gamma_p_baseline:.2e}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Maximum likelihood at γ_p = {gamma_p_values[max_idx]:.2e}")
print(f"Log likelihood value: {ll_gamma_p[max_idx]:.2f}")

### 2b. σ_p Likelihood Curve for Selected Pulsar

In [ ]:
# Use same pulsar index as above
print(f"Creating σ_p likelihood curve for pulsar index {PULSAR_INDEX}")
print(f"Baseline σ_p value for this pulsar: {sigma_p_array[PULSAR_INDEX]:.2e}")

# Define sigma_p parameter range
sigma_p_baseline = sigma_p_array[PULSAR_INDEX]
sigma_p_min = sigma_p_baseline * 0.01  # 1% of baseline
sigma_p_max = sigma_p_baseline * 100   # 100x baseline
sigma_p_values = jnp.logspace(jnp.log10(sigma_p_min), jnp.log10(sigma_p_max), 50)

# Compute likelihood curve
ll_sigma_p = likelihood_curve_sigma_p(sigma_p_values, PULSAR_INDEX)

# Plot
plt.figure(figsize=(10, 6))
plt.semilogx(sigma_p_values, ll_sigma_p, 'm-', linewidth=2)
plt.xlabel(f'σ_p[{PULSAR_INDEX}]')
plt.ylabel('Log Likelihood')
plt.title(f'Likelihood Curve for Pulsar {PULSAR_INDEX} Red Noise σ_p')
plt.grid(True, alpha=0.3)

# Mark maximum and baseline
max_idx = jnp.argmax(ll_sigma_p)
plt.axvline(sigma_p_values[max_idx], color='r', linestyle='--', alpha=0.7,
           label=f'Max at σ_p = {sigma_p_values[max_idx]:.2e}')
plt.axvline(sigma_p_baseline, color='orange', linestyle=':', alpha=0.7,
           label=f'Baseline σ_p = {sigma_p_baseline:.2e}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Maximum likelihood at σ_p = {sigma_p_values[max_idx]:.2e}")
print(f"Log likelihood value: {ll_sigma_p[max_idx]:.2f}")

## 3. Measurement Noise Parameters

### 3a. EFAC Likelihood Curve for Selected Pulsar

In [ ]:
# Use same pulsar index as above
print(f"Creating EFAC likelihood curve for pulsar index {PULSAR_INDEX}")
print(f"Baseline EFAC value for this pulsar: {efac_array[PULSAR_INDEX]:.3f}")

# Define EFAC parameter range
efac_baseline = efac_array[PULSAR_INDEX]
efac_min = max(0.1, efac_baseline * 0.1)  # Don't go below 0.1
efac_max = efac_baseline * 5.0            # 5x baseline
efac_values = jnp.linspace(efac_min, efac_max, 50)

# Compute likelihood curve
ll_efac = likelihood_curve_efac(efac_values, PULSAR_INDEX)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(efac_values, ll_efac, 'c-', linewidth=2)
plt.xlabel(f'EFAC[{PULSAR_INDEX}]')
plt.ylabel('Log Likelihood')
plt.title(f'Likelihood Curve for Pulsar {PULSAR_INDEX} EFAC')
plt.grid(True, alpha=0.3)

# Mark maximum and baseline
max_idx = jnp.argmax(ll_efac)
plt.axvline(efac_values[max_idx], color='r', linestyle='--', alpha=0.7,
           label=f'Max at EFAC = {efac_values[max_idx]:.3f}')
plt.axvline(efac_baseline, color='orange', linestyle=':', alpha=0.7,
           label=f'Baseline EFAC = {efac_baseline:.3f}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Maximum likelihood at EFAC = {efac_values[max_idx]:.3f}")
print(f"Log likelihood value: {ll_efac[max_idx]:.2f}")

### 3b. EQUAD Likelihood Curve for Selected Pulsar

In [ ]:
# Use same pulsar index as above
print(f"Creating EQUAD likelihood curve for pulsar index {PULSAR_INDEX}")
print(f"Baseline EQUAD value for this pulsar: {equad_array[PULSAR_INDEX]:.2e}")

# Define EQUAD parameter range
equad_baseline = equad_array[PULSAR_INDEX]
equad_min = equad_baseline * 0.01  # 1% of baseline
equad_max = equad_baseline * 100   # 100x baseline
equad_values = jnp.logspace(jnp.log10(equad_min), jnp.log10(equad_max), 50)

# Compute likelihood curve
ll_equad = likelihood_curve_equad(equad_values, PULSAR_INDEX)

# Plot
plt.figure(figsize=(10, 6))
plt.semilogx(equad_values, ll_equad, 'y-', linewidth=2)
plt.xlabel(f'EQUAD[{PULSAR_INDEX}]')
plt.ylabel('Log Likelihood')
plt.title(f'Likelihood Curve for Pulsar {PULSAR_INDEX} EQUAD')
plt.grid(True, alpha=0.3)

# Mark maximum and baseline
max_idx = jnp.argmax(ll_equad)
plt.axvline(equad_values[max_idx], color='r', linestyle='--', alpha=0.7,
           label=f'Max at EQUAD = {equad_values[max_idx]:.2e}')
plt.axvline(equad_baseline, color='orange', linestyle=':', alpha=0.7,
           label=f'Baseline EQUAD = {equad_baseline:.2e}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Maximum likelihood at EQUAD = {equad_values[max_idx]:.2e}")
print(f"Log likelihood value: {ll_equad[max_idx]:.2f}")

## 4. Summary and Multi-Parameter Comparison

Create a summary plot showing how the different parameters affect the likelihood.

In [ ]:
# Create a summary figure with all likelihood curves
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Likelihood Curves for All Parameters', fontsize=16)

# GW amplitude
axes[0,0].semilogx(ha_values, ll_ha, 'b-', linewidth=2)
axes[0,0].set_xlabel('GW Amplitude ha')
axes[0,0].set_ylabel('Log Likelihood')
axes[0,0].set_title('GW Amplitude')
axes[0,0].grid(True, alpha=0.3)
max_idx = jnp.argmax(ll_ha)
axes[0,0].axvline(ha_values[max_idx], color='r', linestyle='--', alpha=0.7)

# Gamma_p
axes[0,1].semilogx(gamma_p_values, ll_gamma_p, 'g-', linewidth=2)
axes[0,1].set_xlabel(f'γ_p[{PULSAR_INDEX}] (s⁻¹)')
axes[0,1].set_ylabel('Log Likelihood')
axes[0,1].set_title(f'Pulsar {PULSAR_INDEX} γ_p')
axes[0,1].grid(True, alpha=0.3)
max_idx = jnp.argmax(ll_gamma_p)
axes[0,1].axvline(gamma_p_values[max_idx], color='r', linestyle='--', alpha=0.7)

# Sigma_p
axes[0,2].semilogx(sigma_p_values, ll_sigma_p, 'm-', linewidth=2)
axes[0,2].set_xlabel(f'σ_p[{PULSAR_INDEX}]')
axes[0,2].set_ylabel('Log Likelihood')
axes[0,2].set_title(f'Pulsar {PULSAR_INDEX} σ_p')
axes[0,2].grid(True, alpha=0.3)
max_idx = jnp.argmax(ll_sigma_p)
axes[0,2].axvline(sigma_p_values[max_idx], color='r', linestyle='--', alpha=0.7)

# EFAC
axes[1,0].plot(efac_values, ll_efac, 'c-', linewidth=2)
axes[1,0].set_xlabel(f'EFAC[{PULSAR_INDEX}]')
axes[1,0].set_ylabel('Log Likelihood')
axes[1,0].set_title(f'Pulsar {PULSAR_INDEX} EFAC')
axes[1,0].grid(True, alpha=0.3)
max_idx = jnp.argmax(ll_efac)
axes[1,0].axvline(efac_values[max_idx], color='r', linestyle='--', alpha=0.7)

# EQUAD
axes[1,1].semilogx(equad_values, ll_equad, 'y-', linewidth=2)
axes[1,1].set_xlabel(f'EQUAD[{PULSAR_INDEX}]')
axes[1,1].set_ylabel('Log Likelihood')
axes[1,1].set_title(f'Pulsar {PULSAR_INDEX} EQUAD')
axes[1,1].grid(True, alpha=0.3)
max_idx = jnp.argmax(ll_equad)
axes[1,1].axvline(equad_values[max_idx], color='r', linestyle='--', alpha=0.7)

# Leave last subplot for notes
axes[1,2].text(0.1, 0.5, 
               f'Analysis for Pulsar Index: {PULSAR_INDEX}\n\n'
               f'Total Pulsars: {n_pulsars}\n\n'
               'Red dashed lines show\nmaximum likelihood values\n\n'
               'To analyze different pulsars,\n'
               'change PULSAR_INDEX variable\n'
               'and re-run relevant cells.',
               transform=axes[1,2].transAxes, fontsize=12,
               verticalalignment='center')
axes[1,2].set_xticks([])
axes[1,2].set_yticks([])
axes[1,2].set_title('Analysis Notes')

plt.tight_layout()
plt.show()

## 5. Interactive Parameter Selection

Change the values below to explore different pulsars and parameter ranges.

In [ ]:
def explore_pulsar(pulsar_idx, parameter='sigma_p', n_points=30):
    """Explore likelihood curves for a specific pulsar and parameter.
    
    Parameters:
    -----------
    pulsar_idx : int
        Index of pulsar to analyze (0 to n_pulsars-1)
    parameter : str
        Parameter to vary: 'gamma_p', 'sigma_p', 'efac', or 'equad'
    n_points : int
        Number of points for likelihood curve
    """
    
    if pulsar_idx >= n_pulsars:
        print(f"Error: pulsar_idx must be < {n_pulsars}")
        return
    
    print(f"Analyzing {parameter} for pulsar {pulsar_idx}")
    
    if parameter == 'gamma_p':
        baseline = gamma_p_array[pulsar_idx]
        values = jnp.logspace(jnp.log10(baseline * 0.01), jnp.log10(baseline * 100), n_points)
        ll = likelihood_curve_gamma_p(values, pulsar_idx)
        plt.semilogx(values, ll, linewidth=2)
        plt.xlabel(f'γ_p[{pulsar_idx}] (s⁻¹)')
        
    elif parameter == 'sigma_p':
        baseline = sigma_p_array[pulsar_idx]
        values = jnp.logspace(jnp.log10(baseline * 0.01), jnp.log10(baseline * 100), n_points)
        ll = likelihood_curve_sigma_p(values, pulsar_idx)
        plt.semilogx(values, ll, linewidth=2)
        plt.xlabel(f'σ_p[{pulsar_idx}]')
        
    elif parameter == 'efac':
        baseline = efac_array[pulsar_idx]
        values = jnp.linspace(baseline * 0.1, baseline * 5.0, n_points)
        ll = likelihood_curve_efac(values, pulsar_idx)
        plt.plot(values, ll, linewidth=2)
        plt.xlabel(f'EFAC[{pulsar_idx}]')
        
    elif parameter == 'equad':
        baseline = equad_array[pulsar_idx]
        values = jnp.logspace(jnp.log10(baseline * 0.01), jnp.log10(baseline * 100), n_points)
        ll = likelihood_curve_equad(values, pulsar_idx)
        plt.semilogx(values, ll, linewidth=2)
        plt.xlabel(f'EQUAD[{pulsar_idx}]')
        
    else:
        print(f"Unknown parameter: {parameter}")
        return
    
    plt.ylabel('Log Likelihood')
    plt.title(f'Likelihood Curve for {parameter.upper()} - Pulsar {pulsar_idx}')
    plt.grid(True, alpha=0.3)
    
    # Mark maximum
    max_idx = jnp.argmax(ll)
    plt.axvline(values[max_idx], color='r', linestyle='--', alpha=0.7,
               label=f'Max at {values[max_idx]:.2e}')
    plt.legend()
    plt.show()
    
    print(f"Baseline value: {baseline:.2e}")
    print(f"Maximum likelihood at: {values[max_idx]:.2e}")
    print(f"Log likelihood value: {ll[max_idx]:.2f}")

# Example usage - change these values to explore different parameters
explore_pulsar(pulsar_idx=10, parameter='sigma_p', n_points=40)

In [ ]:
# Quick reference for pulsar information
print("Available pulsars and their indices:")
print("-" * 50)
for i, (idx, row) in enumerate(pulsar_data['metadata'].iterrows()):
    psr_name = row['PSR']
    print(f"Index {i:2d}: {psr_name}")
    if i > 15:  # Limit output
        print(f"... and {n_pulsars - i - 1} more pulsars")
        break